# LLM Cycle-Geometry — Task 15 Full-Model Run (Colab)

Runs Part 1 (circle reproduction, the validation gate) and Part 2 (the AUC comparison ladder) against the real `google/gemma-2-2b` model.

**Before running:** Runtime → Change runtime type → select a GPU (e.g. T4).

## 1. Mount Google Drive and install the package wheel

The wheel (`networkgeometry-0.1.9-py3-none-any.whl`) lives in your Drive folder `NetworkGeometry-Colab/`. This mounts your Drive (one browser authorization click) and installs directly from there — no manual upload needed.

**0.1.9:** the whole shared/neutral pool rewritten to natural, topic-introducing frames that read cleanly for days, months, years and nouns alike, varied by sentence type: `"Tell me about {X}"`, `"This is about {X}"`, `"Have you heard about {X}"`, `"I was just thinking about {X}"`.

**0.1.7:** the Part 2 AUC ladder figure spells out what the **matched** and **specific** contexts are, with real example phrases.

**0.1.6:** correlation matrices at layers 6, 12, 15; strict / paper-exact leg plots the PC1–PC2 **ring manifold** at layers 6/12/15 plus its circularity curve.

**0.1.5:** hierarchy control's bird leaf changed `robin` → `crow`.

**0.1.4 added:** every figure carries a caption defining exactly what it plots, plus state-correlation heatmaps.

**0.1.3 added** the paper-exact "strict leg" and the full Stage-2 comparison ladder.

**0.1.2 fixed a real bug:** templates ended in a period after `{X}`, so extraction read the period's activation, not the state word. Templates are now state-word-final (spec §3.4). **Any results from wheels ≤0.1.1 should be discarded and re-run.**

**When you rebuild the wheel locally after a code fix, the version number bumps (…0.1.8 → 0.1.9 …) so the filename always changes.** Delete the old versioned wheel from the Drive folder and drop in the new one — never overwrite a same-named file, so there's no ambiguity about whether Drive actually has the latest fix.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/NetworkGeometry-Colab'
WHEEL_PATH = f'{DRIVE_DIR}/networkgeometry-0.1.9-py3-none-any.whl'
print('Wheel path:', WHEEL_PATH)

In [ ]:
import numpy, scipy
NUMPY_VER = numpy.__version__
SCIPY_VER = scipy.__version__
print(f'Pinning Colab stock numpy=={NUMPY_VER} scipy=={SCIPY_VER} so nothing upgrades them')

!pip install -q "{WHEEL_PATH}" "numpy=={NUMPY_VER}" "scipy=={SCIPY_VER}" transformer_lens
!pip uninstall -y -q torchvision
!pip uninstall -y -q torchaudio

import importlib.metadata
installed_ver = importlib.metadata.version('networkgeometry')
expected_ver = WHEEL_PATH.split('networkgeometry-')[1].split('-py3')[0]
assert installed_ver == expected_ver, (
    f'Installed networkgeometry=={installed_ver} but expected =={expected_ver} — '
    'the wheel on Drive is stale, re-upload it.'
)
print(f'Confirmed networkgeometry=={installed_ver} installed')

**Root cause of the numpy/scipy crash, actually found:** `pip install transformer_lens` doesn't go through this project's lockfile at all — it's a fresh PyPI resolution every time, independent of the relaxed floors in `pyproject.toml` (those only govern our own wheel's declared requirements, which are already satisfied and never force an upgrade). If `transformer_lens`/`transformers` on PyPI have since bumped their own numpy/scipy floors, installing them can silently upgrade Colab's preinstalled numpy to a version `sklearn` (also preinstalled, imported transitively by `transformers.generation.candidate_generator`) wasn't built against, causing `AttributeError: ... has no attribute '_blas_supports_fpe'`. This is why the fix kept "working" on one VM and breaking again on the next.

The cell above now captures Colab's stock numpy/scipy versions before installing anything, then pins those exact versions in the same `pip install` call as `transformer_lens` — so pip's resolver can't upgrade them no matter what transformer_lens's own metadata requests, keeping Colab's internally-consistent preinstalled numpy/scipy/sklearn trio intact.

Two follow-on version-skew issues also showed up, both fixed by the cell above:

- `torchvision::nms` op-registration failure — Colab's preinstalled `torchvision` doesn't match whatever `torch` gets installed alongside `transformer_lens`. We don't need vision utilities for a text-only Gemma model, so the cell above uninstalls `torchvision` outright.
- `RuntimeError: PyTorch and TorchAudio were compiled with different CUDA versions` — same story for `torchaudio`, which we also don't need. Uninstalled too.

**Important:** any time you change runtime type (GPU, RAM tier, etc.), Colab gives you a brand-new VM with none of this applied — you must re-run the cell above from scratch on that new VM. A plain Runtime to Restart session (no runtime-type change) keeps the same VM/disk, so pip state survives; you'd only need to restart if you still have transformers/torch imported in memory from a prior attempt in this session (stale in-memory "is this package available" caches won't reflect an uninstall until the process restarts).

## 2. Authenticate with Hugging Face

Runs an interactive login widget — paste your token (the same one used locally, or a fresh one). You must have already accepted the license at https://huggingface.co/google/gemma-2-2b under this account.

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
from huggingface_hub import model_info
info = model_info('google/gemma-2-2b')
print('Access confirmed:', info.id, '| gated:', info.gated)

## 3. Load the model (GPU, no-processing loading path, fp16)

The standard `from_pretrained` path (with TransformerLens's weight processing — LayerNorm folding, `center_writing_weights`, `center_unembed`) crashed the whole Colab session with an out-of-memory error during weight conversion, even on GPU — separately from the local Windows/CPU crash. It also can't cleanly apply `center_unembed` to Gemma-2 anyway (TransformerLens warns that `center_unembed` isn't valid with logit softcapping, which Gemma-2 uses).

So this cell uses `from_pretrained_no_processing` (skips the memory-heavy folding/centering steps) with `dtype=torch.float16` (halves peak RAM versus the default float32 — Gemma's checkpoint is natively bfloat16 on disk anyway). Per the documented methodological caveat, note in the findings memo that the no-processing loading path was used: it skips `center_writing_weights`, which zeroes a residual-stream direction that subsequent LayerNorms discard anyway — a small, likely-negligible deviation, but worth recording.

If this still OOMs, the remaining lever is a higher-RAM Colab runtime (Runtime → Change runtime type) — a paid decision, not something to reach for automatically.

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())

from transformer_lens import HookedTransformer
import time
t0 = time.perf_counter()
model = HookedTransformer.from_pretrained_no_processing(
    'google/gemma-2-2b',
    device='cuda' if torch.cuda.is_available() else 'cpu',
    dtype=torch.float16,
)
print(f'Model loaded in {time.perf_counter()-t0:.1f}s')

## 4. Run Part 1 — circle reproduction (the validation gate)

Per spec §4.5: if month/day circularity doesn't reproduce cleanly here, treat it as a pipeline bug and debug before trusting Part 2 — do not proceed on a weak Part 1 result.

In [ ]:
from networkgeometry.run import run_part1
import time

LAYERS = list(range(26))
OUT_DIR = 'results'

t0 = time.perf_counter()
part1_results = run_part1(model, LAYERS, OUT_DIR)
print(f'Part 1 done in {time.perf_counter()-t0:.1f}s')

for name, layer_results in part1_results.items():
    print(f'\n=== {name} ===')
    for lc in layer_results:
        print(f'  layer {lc.layer:2d}: residual={lc.normalized_residual:.4f}  '
              f'angular_order={lc.angular_order:.4f}  top2_var_ratio={lc.top2_variance_ratio:.4f}')

**Stop and inspect before continuing.** Look for layers where `angular_order` and `top2_variance_ratio` are both high (>0.9-ish) for both day and month — those are the clean, reproduced-circle layers. `results/part1_summary.json` now has these numbers on disk too.

## 5. Plot Part 1 manifolds (optional, for visual inspection)

In [ ]:
from networkgeometry.figures.part1_plots import plot_circularity_by_layer
import matplotlib.pyplot as plt
from IPython.display import Image, display

for name, layer_results in part1_results.items():
    path = plot_circularity_by_layer(layer_results, f'{OUT_DIR}/{name}_circularity_by_layer.png')
    display(Image(filename=path))

## 5b. Run Part 1 — strict leg (paper's exact templates)

Per spec §4.3(a): reproduces the paper's own figures literally — one prompt per structure, no template averaging: `"The month of the year is {X}"`, `"The day of the week is {X}"` (day has no paper equivalent; we use the same construction pattern). Years isn't included here yet — it needs a different, not-yet-implemented uncentered/Toeplitz geometry analysis rather than the circular fit used for day/month.

In [ ]:
from networkgeometry.run import run_part1_strict

t0 = time.perf_counter()
part1_strict_results = run_part1_strict(model, LAYERS, OUT_DIR)
print(f'Part 1 (strict leg) done in {time.perf_counter()-t0:.1f}s')

for name, layer_results in part1_strict_results.items():
    print(f'\n=== {name} (strict) ===')
    for lc in layer_results:
        print(f'  layer {lc.layer:2d}: residual={lc.normalized_residual:.4f}  '
              f'angular_order={lc.angular_order:.4f}  top2_var_ratio={lc.top2_variance_ratio:.4f}')

### 5b(ii). Strict-leg plots — the paper's exact-context replication

Two figures per cyclic structure, computed from the paper-exact single template (`"The month of the year is {X}"`, `"The day of the week is {X}"`):

- **PC1–PC2 ring manifold** — the paper's iconic figure: each state projected onto the top 2 principal components; a clean, calendar-ordered ring is the reproduction target. Shown at layers 6, 12, 15. (Polysemous months May/March/August are dropped from the basis, per spec §4.3b.)
- **Circularity vs layer** — angular order and top-2 variance ratio across all 26 layers for the strict template, so you can see which layer's ring is cleanest.

In [ ]:
from networkgeometry.figures.part1_plots import plot_circularity_by_layer, plot_manifold
from networkgeometry.geometry.part1 import manifold_scores
from networkgeometry.stimuli.definitions import load_structures

RING_LAYERS = [6, 12, 15]
structures = load_structures()
strict_mats = mean_state_matrices(model, ['day', 'month'], RING_LAYERS, pool='strict')

for name in ['day', 'month']:
    # the paper's ring, at each representative layer
    labels = strict_mats[name]['labels']
    excluded = structures[name].excluded
    for layer in RING_LAYERS:
        scores, kept = manifold_scores(strict_mats[name]['matrices'][layer], labels, excluded=excluded)
        path = plot_manifold(
            scores, kept, f'{OUT_DIR}/{name}_strict_ring_layer{layer}.png',
            title=f'{name} PC1-PC2 ring, layer {layer} (strict / paper-exact template)')
        display(Image(filename=path))
    # circularity across all layers for the strict template
    path = plot_circularity_by_layer(
        part1_strict_results[name], f'{OUT_DIR}/{name}_strict_circularity_by_layer.png',
        title=f'{name} circularity vs layer (strict / paper-exact template)')
    display(Image(filename=path))

## 5c. State correlation matrices at layers 6, 12, 15

For each of the five structures and each layer, the Pearson correlation matrix between states' residual-stream activation vectors (averaged over the shared-pool templates, all states included). The layer is written into every figure's title, and the caption defines the quantity. For the cyclic structures (day, month) expect **circulant banding** — each state most correlated with its cyclic neighbours; for `hierarchy` expect **nested blocks**, and for `flat` expect no structure beyond the unit diagonal.

In [ ]:
from networkgeometry.run import mean_state_matrices
from networkgeometry.figures.part1_plots import plot_correlation_matrix

CORR_LAYERS = [6, 12, 15]
STRUCTURES = ['day', 'month', 'years', 'hierarchy', 'flat']

corr_mats = mean_state_matrices(model, STRUCTURES, CORR_LAYERS)
for layer in CORR_LAYERS:
    for name in STRUCTURES:
        labels = corr_mats[name]['labels']
        matrix = corr_mats[name]['matrices'][layer]
        path = plot_correlation_matrix(
            matrix, labels, f'{OUT_DIR}/{name}_corr_layer{layer}.png',
            title=f'{name} — state correlation, layer {layer}')
        display(Image(filename=path))

## 6. Run Part 2 — the AUC comparison ladder

Set `LAYERS` below to the gate-passing layers you identified in Part 1 (or leave as the same full range — `run_part2` applies its own within-structure gate internally per layer, so passing all 26 is safe, just slower).

Per spec §5.3, this produces one row per (comparison, layer) covering the full Stage-2 table: `day (within)` / `month (within)` (the gate reference), `day -> month` / `month -> day` in both **matched** and **specific** forms, and the `day -> years` / `day -> hierarchy` / `day -> flat` controls. Cross rows only appear at layers where their source structure passed its own within-structure significance gate.

**matched vs specific** (this is the key distinction the two cross-cycle rows test):
- **matched** = the *shared/neutral* template pool — the **same** frame is used for both structures, so day and month appear in identical wording (e.g. `"Tell me about {X}"`). These frames are deliberately preposition-free so one frame works identically for days, months, years and nouns. This asks: do day and month share a subspace *when the surrounding context is held constant*?
- **specific** = each structure's *own* natural frame, which necessarily differs in wording (day `"We'll meet on {X}"` vs month `"We'll meet in {X}"`). This asks: does the sharing *survive a change of context* — i.e. is it abstract, not phrasing-bound?

In [ ]:
from networkgeometry.run import run_part2

t0 = time.perf_counter()
part2_results = run_part2(model, LAYERS, OUT_DIR)
print(f'Part 2 done in {time.perf_counter()-t0:.1f}s')

for row in sorted(part2_results, key=lambda r: (r.label, r.layer)):
    p_str = (f'raw_p={row.raw_p:.4f} fdr_p={row.fdr_p:.4f} bonferroni_p={row.bonferroni_p:.4f}'
             if row.raw_p is not None else '(within reference, not corrected)')
    print(f'{row.label:28s} layer {row.layer:2d}: auc={row.auc:.4f} sem={row.sem:.4f}  {p_str}')

`results/summary.json` now has the full ladder on disk.

## 7. Plot Part 2 results

One AUC-vs-layer curve per Stage-2 comparison row (spec §5.3 table), chance line at 0.5.

In [ ]:
from networkgeometry.figures.part2_plots import plot_stage2_ladder
from networkgeometry.stimuli.definitions import load_templates

_t = load_templates()
context_note = (
    f"matched = shared/neutral frame used identically for both structures, "
    f"e.g. \"{_t['shared'][0]}\".   "
    f"specific = each structure's own natural frame, "
    f"e.g. day \"{_t['specific']['day'][0]}\", month \"{_t['specific']['month'][0]}\"."
)

path = plot_stage2_ladder(part2_results, f'{OUT_DIR}/stage2_ladder.png', context_note=context_note)
display(Image(filename=path))

## 8. Download results back to your machine

In [ ]:
import shutil
from google.colab import files

shutil.make_archive('task15_results', 'zip', OUT_DIR)
files.download('task15_results.zip')

# Also persist a copy to the same Drive folder
shutil.copy('task15_results.zip', f'{DRIVE_DIR}/task15_results.zip')
print(f'Results also saved to Drive: {DRIVE_DIR}/task15_results.zip')